# Vendor Performance Analytics: End-to-End Project

**Pipeline:** CSV files -> SQLite (`inventory.db`) -> SQL/pandas analysis -> vendor summary table -> Power BI

| Section | Purpose |
|---|---|
| 1 | Setup and batch ingestion into SQLite |
| 2 | Database verification (tables, row counts, schemas) |
| 3 | Exploratory analysis and business SQL queries |
| 4 | Vendor summary table (feature engineering) and export |
| 5 | Summary and next steps for Power BI |

> Folder layout expected: `vendor project/project.ipynb` plus the six CSVs either in `vendor project/data/` or next to the notebook.

In [ ]:
!pip install seaborn

## 1. Setup and Ingestion

In [ ]:
import os
import time
import logging
import sqlite3
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

# ---- Logging: writes to console and to a log file ----
logging.basicConfig(
    filename="ingestion_db.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a",
)
console = logging.StreamHandler()
console.setLevel(logging.INFO)
logging.getLogger().addHandler(console)

In [ ]:
# ---- Configuration ----
DB_PATH = "inventory.db"
DATA_DIR = "data" if os.path.isdir("data") else "."   # falls back to current directory
ENCODING = "latin1"          # the raw files contain non-UTF-8 characters
CHUNK_SIZE = 100_000         # rows per chunk, keeps memory low for big files like sales.csv

EXPECTED_FILES = {
    "begin_inventory.csv", "end_inventory.csv", "purchases.csv",
    "purchase_prices.csv", "sales.csv", "vendor_invoice.csv",
}

engine = create_engine(f"sqlite:///{DB_PATH}")
print(f"Data directory : {os.path.abspath(DATA_DIR)}")
print(f"Database       : {os.path.abspath(DB_PATH)}")

In [ ]:
def read_csv_chunks(path, chunk_size=CHUNK_SIZE):
    """Yield DataFrame chunks, trying latin1 first and falling back to other encodings."""
    for enc in (ENCODING, "utf-8", "cp1252"):
        try:
            # Peek at the first rows to validate the encoding before streaming the file
            pd.read_csv(path, nrows=5, encoding=enc)
            return pd.read_csv(path, chunksize=chunk_size, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            logging.warning(f"Encoding {enc} failed for {path}, trying next.")
    raise UnicodeDecodeError("all", b"", 0, 1, f"No working encoding for {path}")


def ingest_csv_to_db(path, table_name, engine):
    """Stream a CSV into a SQLite table (replace on first chunk, append afterwards)."""
    total_rows = 0
    for i, chunk in enumerate(read_csv_chunks(path)):
        # Light cleaning: strip whitespace from column names
        chunk.columns = chunk.columns.str.strip()
        chunk.to_sql(
            table_name,
            con=engine,
            if_exists="replace" if i == 0 else "append",
            index=False,
        )
        total_rows += len(chunk)
    return total_rows


def load_raw_data(data_dir=DATA_DIR):
    """Scan the directory and ingest every CSV into SQLite with error handling."""
    start = time.time()
    found = sorted(f for f in os.listdir(data_dir) if f.lower().endswith(".csv"))

    if not found:
        raise FileNotFoundError(f"No CSV files found in '{data_dir}'")

    missing = EXPECTED_FILES - set(found)
    if missing:
        logging.warning(f"Expected files not found: {sorted(missing)}")

    summary = []
    for file in found:
        path = os.path.join(data_dir, file)
        table_name = os.path.splitext(file)[0]
        try:
            t0 = time.time()
            rows = ingest_csv_to_db(path, table_name, engine)
            elapsed = time.time() - t0
            logging.info(f"Ingested {file:<22} -> {table_name:<18} {rows:>10,} rows in {elapsed:6.1f}s")
            summary.append({"file": file, "table": table_name, "rows": rows,
                            "seconds": round(elapsed, 1), "status": "OK"})
        except Exception as e:
            logging.error(f"Failed to ingest {file}: {e}")
            summary.append({"file": file, "table": table_name, "rows": 0,
                            "seconds": 0, "status": f"FAILED: {e}"})

    logging.info(f"----- Ingestion complete in {(time.time() - start) / 60:.2f} minutes -----")
    return pd.DataFrame(summary)

ingestion_report = load_raw_data()
ingestion_report

In [ ]:
with engine.begin() as conn:
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_purchases_vendor_brand ON purchases (VendorNumber, Brand)"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_sales_vendor_brand ON sales (VendorNo, Brand)"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_invoice_vendor ON vendor_invoice (VendorNumber)"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_prices_brand ON purchase_prices (Brand)"))
    conn.execute(text("ANALYZE"))
print("Indexes created")

## 2. Database Verification

In [ ]:
# Helper to run SQL and get a DataFrame back
def run_query(sql, params=None):
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn, params=params)

# 2.1 Tables in the database
tables = run_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables

In [ ]:
# 2.2 Row counts per table
counts = []
for t in tables["name"]:
    n = run_query(f"SELECT COUNT(*) AS n FROM {t}")["n"][0]
    counts.append({"table": t, "row_count": n})
pd.DataFrame(counts)

In [ ]:
# 2.3 Schema of every table (column names, types)
for t in tables["name"]:
    print(f"\n=== {t} ===")
    display(run_query(f"PRAGMA table_info({t})")[["name", "type"]].T)

In [ ]:
# 2.4 Peek at a few rows from each table
for t in tables["name"]:
    print(f"\n=== {t} ===")
    display(run_query(f"SELECT * FROM {t} LIMIT 3"))

## 3. Exploratory Data Analysis and Business SQL Queries

> The queries below assume the standard column names for this dataset (`VendorNumber`, `Brand`, `PurchasePrice`, `SalesDollars`, ...). If any query errors with "no such column", compare against the schema printed in Section 2 and adjust.

### 3.1 Data quality checks

In [ ]:
# Memory-safe null and data quality checks per table
for t in tables["name"]:
    # Get total row count first
    total_rows = run_query(f"SELECT COUNT(*) AS n FROM {t}")["n"][0]
    print(f"\n=== {t} | Total Rows: {total_rows:,} ===")
    
    # For large tables, check nulls using SQL aggregation instead of loading all rows into RAM
    null_check_query = f"""
        SELECT 
            SUM(CASE WHEN '' THEN 1 ELSE 0 END) -- placeholder safety
        FROM {t}
    """
    # Alternatively, sample or inspect columns efficiently:
    schema = run_query(f"PRAGMA table_info({t})")
    nulls_dict = {}
    for col in schema["name"]:
        null_count = run_query(f"SELECT COUNT(*) AS n FROM {t} WHERE [{col}] IS NULL")["n"][0]
        if null_count > 0:
            nulls_dict[col] = null_count
            
    print("Columns with nulls:", nulls_dict or "none")

### 3.2 Vendor performance

In [ ]:
# Top 10 vendors by total purchase spend
top_vendors_purchases = run_query("""
SELECT VendorNumber,
       VendorName,
       COUNT(DISTINCT PONumber)       AS TotalOrders,
       SUM(Quantity)                  AS TotalQuantity,
       ROUND(SUM(Dollars), 2)         AS TotalPurchaseDollars,
       ROUND(AVG(PurchasePrice), 2)   AS AvgUnitPrice
FROM purchases
GROUP BY VendorNumber, VendorName
ORDER BY TotalPurchaseDollars DESC
LIMIT 10
""")
top_vendors_purchases

In [ ]:
# Purchase concentration: how much of total spend is the top 10 vendors?
total_spend = run_query("SELECT SUM(Dollars) AS s FROM purchases")["s"][0]
share = top_vendors_purchases["TotalPurchaseDollars"].sum() / total_spend
print(f"Top 10 vendors account for {share:.1%} of total purchase spend")

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=top_vendors_purchases, y="VendorName", x="TotalPurchaseDollars", ax=ax, color="#4C72B0")
ax.set_title("Top 10 Vendors by Purchase Spend")
ax.set_xlabel("Total Purchase Dollars")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Delivery lead time and payment terms per vendor (PO date -> receiving date -> pay date)
lead_times = run_query("""
SELECT VendorNumber,
       VendorName,
       COUNT(*) AS Lines,
       ROUND(AVG(julianday(ReceivingDate) - julianday(PODate)), 1) AS AvgLeadTimeDays,
       ROUND(MAX(julianday(ReceivingDate) - julianday(PODate)), 0) AS MaxLeadTimeDays
FROM purchases
WHERE ReceivingDate IS NOT NULL AND PODate IS NOT NULL
GROUP BY VendorNumber, VendorName
HAVING Lines >= 50
ORDER BY AvgLeadTimeDays DESC
LIMIT 15
""")
lead_times

In [ ]:
# Freight cost vs invoice value per vendor (from vendor_invoice)
freight = run_query("""
SELECT VendorNumber,
       VendorName,
       COUNT(*)                         AS Invoices,
       ROUND(SUM(Dollars), 2)           AS InvoiceDollars,
       ROUND(SUM(Freight), 2)           AS FreightCost,
       ROUND(100.0 * SUM(Freight) / SUM(Dollars), 2) AS FreightPctOfInvoice
FROM vendor_invoice
GROUP BY VendorNumber, VendorName
HAVING InvoiceDollars > 0
ORDER BY FreightPctOfInvoice DESC
LIMIT 15
""")
freight

### 3.3 Purchases vs. sales

In [ ]:
# Purchase vs sales comparison by brand (the core "are we making money" view)
brand_ps = run_query("""
WITH p AS (
    SELECT Brand,
           SUM(Quantity) AS PurchasedQty,
           SUM(Dollars)  AS PurchasedDollars
    FROM purchases
    GROUP BY Brand
),
s AS (
    SELECT Brand,
           SUM(SalesQuantity) AS SoldQty,
           SUM(SalesDollars)  AS SalesDollars
    FROM sales
    GROUP BY Brand
)
SELECT p.Brand,
       p.PurchasedQty,
       COALESCE(s.SoldQty, 0)                           AS SoldQty,
       ROUND(p.PurchasedDollars, 2)                     AS PurchasedDollars,
       ROUND(COALESCE(s.SalesDollars, 0), 2)            AS SalesDollars,
       ROUND(COALESCE(s.SalesDollars, 0) - p.PurchasedDollars, 2) AS GrossMargin
FROM p
LEFT JOIN s ON p.Brand = s.Brand
ORDER BY GrossMargin DESC
""")
print("Top 10 brands by gross margin")
display(brand_ps.head(10))
print("Bottom 10 brands by gross margin (loss makers)")
display(brand_ps.tail(10))

In [ ]:
# Monthly sales trend
monthly_sales = run_query("""
SELECT strftime('%Y-%m', SalesDate) AS Month,
       ROUND(SUM(SalesDollars), 2)  AS SalesDollars,
       SUM(SalesQuantity)           AS UnitsSold
FROM sales
GROUP BY Month
ORDER BY Month
""")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly_sales["Month"], monthly_sales["SalesDollars"], marker="o")
ax.set_title("Monthly Sales Dollars")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()
monthly_sales

### 3.4 Inventory turnover

In [ ]:
# Inventory snapshot: begin vs end on-hand units and value, per brand
inventory_change = run_query("""
WITH b AS (
    SELECT Brand,
           SUM(onHand)         AS BeginUnits,
           SUM(onHand * Price) AS BeginValue
    FROM begin_inventory
    GROUP BY Brand
),
e AS (
    SELECT Brand,
           SUM(onHand)         AS EndUnits,
           SUM(onHand * Price) AS EndValue
    FROM end_inventory
    GROUP BY Brand
)
SELECT COALESCE(b.Brand, e.Brand)                  AS Brand,
       COALESCE(b.BeginUnits, 0)                   AS BeginUnits,
       COALESCE(e.EndUnits, 0)                     AS EndUnits,
       ROUND(COALESCE(b.BeginValue, 0), 2)         AS BeginValue,
       ROUND(COALESCE(e.EndValue, 0), 2)           AS EndValue
FROM b
LEFT JOIN e ON b.Brand = e.Brand
""")
inventory_change.head(10)

In [ ]:
# Inventory turnover = COGS / average inventory value
# COGS is approximated by purchase dollars (the standard simplification for this dataset)
turnover = run_query("""
WITH inv AS (
    SELECT b.Brand,
           (SUM(b.onHand * b.Price) + COALESCE(e.EndValue, 0)) / 2.0 AS AvgInventoryValue
    FROM begin_inventory b
    LEFT JOIN (SELECT Brand, SUM(onHand * Price) AS EndValue
               FROM end_inventory GROUP BY Brand) e
           ON b.Brand = e.Brand
    GROUP BY b.Brand
),
cogs AS (
    SELECT Brand, SUM(Dollars) AS COGS
    FROM purchases
    GROUP BY Brand
)
SELECT cogs.Brand,
       ROUND(cogs.COGS, 2)              AS COGS,
       ROUND(inv.AvgInventoryValue, 2)  AS AvgInventoryValue,
       ROUND(cogs.COGS / NULLIF(inv.AvgInventoryValue, 0), 2) AS InventoryTurnover
FROM cogs
JOIN inv ON cogs.Brand = inv.Brand
WHERE inv.AvgInventoryValue > 0
ORDER BY InventoryTurnover ASC
""")

print("Slowest-moving brands (lowest turnover) -> candidates for reducing stock")
display(turnover.head(10))
print("Fastest-moving brands (highest turnover)")
display(turnover.tail(10))
print(f"Median turnover across brands: {turnover['InventoryTurnover'].median():.2f}")

### 3.5 Top-performing products and vendors (by sales)

In [ ]:
top_products = run_query("""
SELECT Brand,
       Description,
       SUM(SalesQuantity)          AS UnitsSold,
       ROUND(SUM(SalesDollars), 2) AS SalesDollars
FROM sales
GROUP BY Brand, Description
ORDER BY SalesDollars DESC
LIMIT 10
""")
top_products

In [ ]:
top_vendors_sales = run_query("""
SELECT VendorNo,
       VendorName,
       SUM(SalesQuantity)          AS UnitsSold,
       ROUND(SUM(SalesDollars), 2) AS SalesDollars
FROM sales
GROUP BY VendorNo, VendorName
ORDER BY SalesDollars DESC
LIMIT 10
""")
top_vendors_sales

In [ ]:
# Sales by store (which locations drive revenue?)
store_sales = run_query("""
SELECT Store,
       ROUND(SUM(SalesDollars), 2) AS SalesDollars,
       SUM(SalesQuantity)          AS UnitsSold
FROM sales
GROUP BY Store
ORDER BY SalesDollars DESC
LIMIT 10
""")
store_sales

## 4. Vendor Summary Table (Feature Engineering)

Combine purchases, sales and freight into **one row per vendor-brand**. This single table feeds the dashboard and keeps Power BI fast and simple.

In [ ]:
vendor_sales_summary = run_query("""
WITH FreightSummary AS (
    SELECT VendorNumber,
           SUM(Freight) AS FreightCost
    FROM vendor_invoice
    GROUP BY VendorNumber
),
PurchaseSummary AS (
    SELECT p.VendorNumber,
           p.VendorName,
           p.Brand,
           p.Description,
           p.PurchasePrice,
           pp.Volume,
           pp.Price AS ActualPrice,
           SUM(p.Quantity) AS TotalPurchaseQuantity,
           SUM(p.Dollars)  AS TotalPurchaseDollars
    FROM purchases p
    JOIN purchase_prices pp ON p.Brand = pp.Brand
    WHERE p.PurchasePrice > 0
    GROUP BY p.VendorNumber, p.VendorName, p.Brand, p.Description, p.PurchasePrice, pp.Price, pp.Volume
),
SalesSummary AS (
    SELECT VendorNo,
           Brand,
           SUM(SalesDollars)  AS TotalSalesDollars,
           SUM(SalesPrice)    AS TotalSalesPrice,
           SUM(SalesQuantity) AS TotalSalesQuantity,
           SUM(ExciseTax)     AS TotalExciseTax
    FROM sales
    GROUP BY VendorNo, Brand
)
SELECT ps.VendorNumber,
       ps.VendorName,
       ps.Brand,
       ps.Description,
       ps.PurchasePrice,
       ps.ActualPrice,
       ps.Volume,
       ps.TotalPurchaseQuantity,
       ps.TotalPurchaseDollars,
       ss.TotalSalesQuantity,
       ss.TotalSalesDollars,
       ss.TotalSalesPrice,
       ss.TotalExciseTax,
       fs.FreightCost
FROM PurchaseSummary ps
LEFT JOIN SalesSummary ss
       ON ps.VendorNumber = ss.VendorNo AND ps.Brand = ss.Brand
LEFT JOIN FreightSummary fs
       ON ps.VendorNumber = fs.VendorNumber
ORDER BY ps.TotalPurchaseDollars DESC
""")
print(vendor_sales_summary.shape)
vendor_sales_summary.head()

In [ ]:
# ---- Cleaning ----
df = vendor_sales_summary.copy()

df["Volume"] = pd.to_numeric(df["Volume"], errors="coerce")
df = df.fillna(0)                                            # unsold brands -> 0 sales
for c in ("VendorName", "Description"):
    df[c] = df[c].astype(str).str.strip()

# ---- Derived KPIs ----
df["GrossProfit"]            = df["TotalSalesDollars"] - df["TotalPurchaseDollars"]
df["ProfitMargin"]           = np.where(df["TotalSalesDollars"] > 0,
                                        df["GrossProfit"] / df["TotalSalesDollars"] * 100, 0)
df["StockTurnover"]          = np.where(df["TotalPurchaseQuantity"] > 0,
                                        df["TotalSalesQuantity"] / df["TotalPurchaseQuantity"], 0)
df["SalesToPurchaseRatio"]   = np.where(df["TotalPurchaseDollars"] > 0,
                                        df["TotalSalesDollars"] / df["TotalPurchaseDollars"], 0)
df["UnsoldInventoryValue"]   = (df["TotalPurchaseQuantity"] - df["TotalSalesQuantity"]) * df["PurchasePrice"]

vendor_summary = df
vendor_summary.describe().T

In [ ]:
# ---- Persist back to SQLite and export CSV for Power BI ----
vendor_summary.to_sql("vendor_sales_summary", con=engine, if_exists="replace", index=False)
vendor_summary.to_csv("vendor_sales_summary.csv", index=False)
print("Saved table 'vendor_sales_summary' and file 'vendor_sales_summary.csv'")

### 4.1 Key business questions answered from the summary

In [ ]:
# Brands needing promotional or pricing action (low sales, high margin)
brand_perf = (vendor_summary.groupby("Description")
              .agg(TotalSales=("TotalSalesDollars", "sum"), Margin=("ProfitMargin", "mean"))
              .reset_index())
low_sales_thr = brand_perf["TotalSales"].quantile(0.15)
high_margin_thr = brand_perf["Margin"].quantile(0.85)

target_brands = brand_perf[(brand_perf["TotalSales"] <= low_sales_thr) &
                           (brand_perf["Margin"] >= high_margin_thr)]
print(f"{len(target_brands)} brands have low sales but high margin (promotion candidates)")
target_brands.sort_values("TotalSales").head(10)

In [ ]:
# Top 10 vendors vs. rest: share of purchases
vendor_totals = (vendor_summary.groupby("VendorName")["TotalPurchaseDollars"]
                 .sum().sort_values(ascending=False))
top10 = vendor_totals.head(10)
print(f"Top 10 vendors: {top10.sum() / vendor_totals.sum():.1%} of purchase dollars")

fig, ax = plt.subplots(figsize=(8, 8))
labels = list(top10.index) + ["All other vendors"]
sizes = list(top10.values) + [vendor_totals.iloc[10:].sum()]
ax.pie(sizes, labels=labels, autopct="%1.1f%%", startangle=140, pctdistance=0.85)
ax.set_title("Purchase Contribution: Top 10 Vendors vs. Others")
plt.show()

In [ ]:
# Does buying in bulk lower unit cost?
df_bulk = vendor_summary[vendor_summary["TotalPurchaseQuantity"] > 0].copy()
df_bulk["UnitPurchasePrice"] = df_bulk["TotalPurchaseDollars"] / df_bulk["TotalPurchaseQuantity"]
df_bulk["OrderSize"] = pd.qcut(df_bulk["TotalPurchaseQuantity"], q=3, labels=["Small", "Medium", "Large"])

print(df_bulk.groupby("OrderSize")["UnitPurchasePrice"].mean().round(2))

sns.boxplot(data=df_bulk, x="OrderSize", y="UnitPurchasePrice")
plt.title("Unit Purchase Price by Order Size")
plt.show()

In [ ]:
# Vendors with the most capital tied up in unsold inventory
unsold = (vendor_summary.groupby("VendorName")["UnsoldInventoryValue"]
          .sum().sort_values(ascending=False).head(10))
print(f"Total unsold inventory value: ${vendor_summary['UnsoldInventoryValue'].sum():,.0f}")
unsold.to_frame("UnsoldInventoryValue")

In [ ]:
# Statistical check: do top and low performing vendors differ in margin?
from scipy import stats

vendor_perf = vendor_summary.groupby("VendorName")["TotalSalesDollars"].sum()
top_set = vendor_summary[vendor_summary["VendorName"].isin(vendor_perf[vendor_perf >= vendor_perf.quantile(0.75)].index)]
low_set = vendor_summary[vendor_summary["VendorName"].isin(vendor_perf[vendor_perf <= vendor_perf.quantile(0.25)].index)]

top_m = top_set["ProfitMargin"].replace([np.inf, -np.inf], np.nan).dropna()
low_m = low_set["ProfitMargin"].replace([np.inf, -np.inf], np.nan).dropna()

t, p = stats.ttest_ind(top_m, low_m, equal_var=False)
print(f"Top-vendor mean margin: {top_m.mean():.2f}%   Low-vendor mean margin: {low_m.mean():.2f}%")
print(f"Welch t-test: t={t:.2f}, p={p:.4g} -> {'significant' if p < 0.05 else 'not significant'} at 5%")

## 5. Summary and Next Steps

### What this notebook produced
- `inventory.db` with the six raw tables plus a curated **`vendor_sales_summary`** table.
- `vendor_sales_summary.csv` for direct import into BI tools.
- `ingestion_db.log` as an audit trail of the load.

### Findings to look for (check the outputs above)
- **Concentration risk:** share of spend held by the top 10 vendors.
- **Slow movers:** brands with very low inventory turnover tie up cash.
- **Loss makers:** brands with negative gross margin in Section 3.3.
- **Promotion candidates:** low sales and high margin brands.
- **Freight:** vendors where freight is a high percentage of invoice value.

### Moving to Power BI
1. **Get Data** -> *Text/CSV* -> `vendor_sales_summary.csv`. For the live database, install the SQLite ODBC driver and use *Get Data -> ODBC*.
2. In Power Query, confirm types: IDs as Text, dollars and quantities as Decimal, `ProfitMargin` as Percentage (divide by 100 or format).
3. **DAX measures** to create:
   ```
   Total Sales      = SUM(vendor_sales_summary[TotalSalesDollars])
   Total Purchases  = SUM(vendor_sales_summary[TotalPurchaseDollars])
   Gross Profit     = [Total Sales] - [Total Purchases]
   Profit Margin %  = DIVIDE([Gross Profit], [Total Sales])
   Stock Turnover   = DIVIDE(SUM(vendor_sales_summary[TotalSalesQuantity]), SUM(vendor_sales_summary[TotalPurchaseQuantity]))
   Unsold Inventory = SUM(vendor_sales_summary[UnsoldInventoryValue])
   ```
4. **Suggested dashboard layout:**
   - KPI cards: Total Sales, Total Purchases, Gross Profit, Profit Margin %, Unsold Inventory.
   - Bar chart: Top 10 vendors by sales; Top 10 brands by sales.
   - Donut: Purchase contribution by vendor.
   - Scatter: Profit Margin vs. Total Sales by brand (find promotion candidates).
   - Table with conditional formatting: low-turnover brands.
   - Slicers: Vendor, Brand, Description.
5. Publish to Power BI Service and schedule a refresh if the source CSV is regenerated.

### Possible extensions
- Replace `if_exists="replace"` with incremental loads.
- Add demand forecasting (e.g., Prophet or SARIMA) on monthly sales.
- Wrap the notebook logic into `ingestion_db.py` and `get_vendor_summary.py` scripts and schedule them.